# Synthetic data generation using VAEs

## 1. Introduction


<img src="./data/VAE_Basic.png" alt="VAE Structure" width="600"/>


#### 🔍 What is a VAE?

A **Variational Autoencoder** is a **generative model** that learns how to compress data into a **latent space** and then **reconstruct** it. Unlike traditional autoencoders, VAEs are **probabilistic** — they model distributions rather than just fixed values.

---

#### 🧠 How It Work?

A VAE has two main parts:

1. **Encoder**: Compresses the input `x` into a probability distribution over latent variables `z`, specifically a **mean** (μ) and **standard deviation** (σ).
2. **Decoder**: Samples a latent vector `z` from this distribution and reconstructs the input `x'`.

---

* We assume the latent space follows a **normal distribution**:

  $$
  z \sim \mathcal{N}(\mu(x), \sigma(x)^2)
  $$

* The **loss function** combines:

  1. **Reconstruction loss** (how well the output matches the input), e.g. Mean Squared Error (MSE).
  2. **KL divergence** between the learned distribution and a standard normal distribution $\mathcal{N}(0, 1)$:

  $$
  \mathcal{L} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - \text{KL}(q(z|x) \parallel p(z))
  $$

  This encourages the latent space to be smooth and continuous.

---

#### ✅ Why It Matters?

* VAEs are great for **generating new data**, **dimensionality reduction**, and learning **structured latent representations**.
* Unlike standard autoencoders, VAEs **can generate realistic and varied samples** because of their probabilistic nature.

---



## 2. SDV (Synthetic Data Vault)

It's an MIT package for creating sythetic data

### A. Import the data and packages

In [58]:
from sdv.metadata import SingleTableMetadata # metadata extractor
from sdv.single_table import TVAESynthesizer # VAE data synthesizer 
import pprint # prints dictionaries nicely
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score,confusion_matrix,accuracy_score,classification_report

In [5]:
# this data was imported from https://www.kaggle.com/datasets/uciml/german-credit
df = pd.read_csv("./data/german_credit_data.csv", index_col=0)

In [6]:
df.describe(include='all').T.sort_values("freq")

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Checking account,606,3,little,274,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Purpose,1000,8,car,337,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Saving accounts,817,4,little,603,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,1000,2,male,690,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Housing,1000,3,own,713,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1000.0,NaN,NaN,NaN,35.546,11.375469,19.0,27.0,33.0,42.0,75.0
Job,1000.0,NaN,NaN,NaN,1.904,0.653614,0.0,2.0,2.0,2.0,3.0
Credit amount,1000.0,NaN,NaN,NaN,3271.258,2822.736876,250.0,1365.5,2319.5,3972.25,18424.0
Duration,1000.0,NaN,NaN,NaN,20.903,12.058814,4.0,12.0,18.0,24.0,72.0


In [7]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)
pprint.pprint(metadata.to_dict())

{'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1',
 'columns': {'Age': {'sdtype': 'numerical'},
             'Checking account': {'sdtype': 'categorical'},
             'Credit amount': {'sdtype': 'numerical'},
             'Duration': {'sdtype': 'numerical'},
             'Housing': {'sdtype': 'categorical'},
             'Job': {'sdtype': 'categorical'},
             'Purpose': {'sdtype': 'categorical'},
             'Saving accounts': {'sdtype': 'categorical'},
             'Sex': {'sdtype': 'categorical'}}}


In [8]:
# update the types
metadata.update_column(column_name="Saving accounts",
sdtype="categorical")

In [9]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)
pprint.pprint(metadata.to_dict())

{'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1',
 'columns': {'Age': {'sdtype': 'numerical'},
             'Checking account': {'sdtype': 'categorical'},
             'Credit amount': {'sdtype': 'numerical'},
             'Duration': {'sdtype': 'numerical'},
             'Housing': {'sdtype': 'categorical'},
             'Job': {'sdtype': 'categorical'},
             'Purpose': {'sdtype': 'categorical'},
             'Saving accounts': {'sdtype': 'categorical'},
             'Sex': {'sdtype': 'categorical'}}}


### B. Train a VAE to generate sythetic data

In [12]:
synthesizer = TVAESynthesizer(metadata, epochs=10000,verbose=True)
synthesizer.fit(df)
synthetic_data = synthesizer.sample(num_rows=10000)

/home/sako/anaconda3/lib/python3.11/site-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
Loss: -5.670: 100%|███████████████████████| 10000/10000 [09:33<00:00, 17.45it/s]


In [14]:
synthetic_data.describe(include='all').T.sort_values("freq")

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Checking account,5827,3,little,2693,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Purpose,10000,8,car,3047,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Saving accounts,8278,4,little,6067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,10000,2,male,7100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Housing,10000,3,own,7551,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,10000.0,NaN,NaN,NaN,36.0616,11.535273,19.0,27.0,33.0,43.0,75.0
Job,10000.0,NaN,NaN,NaN,1.9079,0.60684,0.0,2.0,2.0,2.0,3.0
Credit amount,10000.0,NaN,NaN,NaN,3136.5074,2470.42188,250.0,1442.0,2334.5,3880.25,17287.0
Duration,10000.0,NaN,NaN,NaN,20.482,11.659283,4.0,12.0,17.0,24.0,65.0


### C. Validate results uning a Random forrest classifier

In [25]:
# stack synthetic and real data 
X = pd.concat([synthetic_data, df])
X.index=range(len(X))


In [62]:
# one-hot-encoding of categorical data
categorical_columns = ["Sex", "Housing", "Saving accounts","Checking account", "Purpose"]
X_encoded = pd.get_dummies(X, columns=categorical_columns)

In [63]:
X.sample(5)

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
9322,61,male,2,own,NaN,NaN,3171,31,furniture/equipment
6350,27,male,1,own,little,NaN,2407,28,business
9257,54,female,0,own,little,little,1191,9,car
8730,38,male,2,own,little,NaN,1826,18,furniture/equipment
1194,26,male,2,own,NaN,NaN,1481,29,radio/TV


In [64]:
X_encoded.sample(5).T

,3025,1226,7595,6342,3684
Age,35,39,24,33,28
Job,3,1,1,3,2
Credit amount,1583,2876,3209,3136,6679
Duration,8,15,13,14,24
Sex_female,True,False,False,True,True
Sex_male,False,True,True,False,False
Housing_free,True,False,False,False,False
Housing_own,False,True,True,True,False
Housing_rent,False,False,False,False,True
Saving accounts_little,True,True,True,True,True


In [65]:
# add labels for real and synthetic ones
y = [0] * len(synthetic_data) + [1] * len(df)

In [32]:
# use random forrest classifier and CV with 5 folds
model = RandomForestClassifier()
cv_preds = cross_val_predict(model,
                             X_encoded,
                             y,
                             cv=5,
                             n_jobs=-1,
                             method="predict_proba")

In [67]:
# Estimate results for training results 
roc_adv_score = roc_auc_score(y_true=y, y_score=cv_preds[:, 1])
print(f"roc auc adv score: {roc_adv_score:0.3f}")

roc auc adv score: 0.566


#### 📊 ROC AUC Score Summary

* **ROC AUC (Receiver Operating Characteristic - Area Under the Curve)** is a metric used for **binary classification** to evaluate a model's ability to distinguish between positive and negative cases.
* **Score range**:

  * **1.0** → perfect classifier
  * **0.5** → random guessing
* A **ROC AUC score of \~0.567** indicates the model **barely outperforms random guessing**, suggesting the **synthetic data closely resembles original data**.
* Learn more at: [MLU ROC AUC Explanation](https://mlu-explain.github.io/roc-auc/) — an Amazon initiative to teach ML concepts.
* The following section analyzes real tabular datasets from **UCI** and **Kaggle**, identifying real-world issues and applying fixes.

To see a review about classification metrics like Precisio, Recall, and the ROC-AUC  **[go to this notebook](./extra_Classification_Metrics.ipynb)**